In [1]:
import pandas as pd
import numpy as np
from dl_client import DatalakeClient

client = DatalakeClient()

In [2]:
search = client.query_files(
    query={'custom.level' : 'raw'})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(zip_files.keys())

dict_keys(['ADNIMERGE_06Jun2025.csv', 'ADSP_PHC_BIOMARKER_06Jun2025.csv', 'BLCHANGE_06Jun2025.csv', 'DXSUM_06Jun2025.csv', 'MMSE_06Jun2025.csv', 'NEUROPATH_06Jun2025.csv', 'PTDEMOG_06Jun2025.csv'])


c:\Users\ChiaraPollicini\anaconda3\envs\aind101\lib\site-packages\dl_client\client.py:490: DtypeWarning: Columns (19,20,21,50,51,104,105,106) have mixed types. Specify dtype option on import or set low_memory=False.
  file_contents[filename] = pd_local.read_csv(io.BytesIO(file_content), delimiter=delimiter)


In [3]:
def select_variables(df, file_name, support_file, type='raw/'):
    metadata = client.get_metadata(
        object_name = type+file_name
    )

    file_code = metadata['metadata']['custom']['file_code']
    
    support_file = support_file[support_file['file_code']==file_code]
    lst_variable = [x for x in support_file['variable_code'].unique() if x in list(df.columns)] 
    df_new = df[lst_variable]

    return df_new

In [60]:
def check_missing_values(df, key, stampa=False):
    n_missing = int(df[key].isna().sum())
    n_tot = int(df.shape[0])
    n_valid = int(n_tot - n_missing)
    
    pop = df[key].unique().tolist()
    pop_valid = df[df[key].isna() == False][key].unique().tolist()
    pop_missing = [x for x in pop if x not in pop_valid]
    if len(pop_missing) == 0:
        pop_missing = [None]

    if stampa:
        print(f'Missing/total values:        {n_missing}/{n_tot}\nValid/totalvalues:          {n_valid}/{n_tot}')
        print('missing population:  ', pop_missing)
    
    return n_tot, n_valid, n_missing, pop_valid, pop_missing

In [46]:
def check_type_range_variables(df, key):
    n = df[key].first_valid_index()
    tipo = type(df[key][n])
    options = df[key].unique()
    if tipo is not str:
        intervallo = [float(df[key].max()), float(df[key].min())]
    else:
        intervallo = [None]
    
    if len(options) <= 10:
        classes = options
    else:
        classes = [None]

    return tipo, intervallo, classes

In [ ]:
def get_varible_info(df, key, file_code, support_file):
    _, n_valid, n_missing, _, pop_missing = check_missing_values(df, key)
    tipo, intervallo, classes = check_type_range_variables(df, key)

    index = support_file.index[(support_file['file_code'] == file_code) & (support_file['variable_code'] == key)][0]

    support_file['type_variable'][index] = tipo
    support_file['classes'][index] = ', '.join(map(str, classes))
    support_file['range'][index] = ', '.join(map(str, intervallo))
    support_file['valid_values'][index] = int(n_valid)
    support_file['missing_values'][index] = int(n_missing)
    support_file['missing_pop'][index] = ', '.join(map(str, pop_missing))

    return support_file

In [ ]:
file_name = 'ADNIMERGE_06Jun2025.csv'
df = zip_files[file_name]

support_file = pd.read_excel('ADNI_variables_statistics.xlsx')
df_new = select_variables(df, file_name, support_file)

In [ ]:
file_code = 'ADNIMERGE'
for key in df_new.keys():
    get_varible_info(df_new, key, file_code, support_file)

C:\Users\ChiaraPollicini\AppData\Local\Temp\ipykernel_20036\2846132453.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  support_file['type_variable'][index] = tipo
C:\Users\ChiaraPollicini\AppData\Local\Temp\ipykernel_20036\2846132453.py:

In [65]:
support_file

,file_name,file_code,parameter,population,variable_code,type_variable,classes,range,valid_values,missing_values,missing_pop
0,Key ADNI tables merged into one table,ADNIMERGE,ID,"1,GO,2,3",PTID,<class 'str'>,None,None,16421.0,0.0,None
1,Key ADNI tables merged into one table,ADNIMERGE,ID,"1,GO,2,3",RID,<class 'numpy.int64'>,None,"7125.0, 2.0",16421.0,0.0,None
2,Key ADNI tables merged into one table,ADNIMERGE,Cohorte,"1,GO,2,3",COLPROT,<class 'str'>,"ADNI1, ADNI2, ADNIGO, ADNI3",None,16421.0,0.0,None
3,Key ADNI tables merged into one table,ADNIMERGE,visit,"1,GO,2,3",VISCODE,<class 'str'>,None,None,16421.0,0.0,None
4,Key ADNI tables merged into one table,ADNIMERGE,visit,"1,GO,2,3",EXAMDATE,<class 'str'>,None,None,16421.0,0.0,None
...,...,...,...,...,...,...,...,...,...,...,...
724,Diagnostic Summary,DXSUM,Other,4,DXMPTR5,NaN,NaN,NaN,NaN,NaN,NaN
725,Diagnostic Summary,DXSUM,Other,4,DXMPTR6,NaN,NaN,NaN,NaN,NaN,NaN
726,Diagnostic Summary,DXSUM,Other,4,DXPARK,NaN,NaN,NaN,NaN,NaN,NaN
727,Diagnostic Summary,DXSUM,ID,4,ID,NaN,NaN,NaN,NaN,NaN,NaN


In [71]:
dd_boh = df.apply(lambda x: x.ORIGPROT == x.COLPROT, axis=1)

In [75]:
dd_boh[dd_boh == False].index

Index([  110,   126,   158,   168,   239,   278,   301,   315,   320,   334,
       ...
       16388, 16394, 16395, 16400, 16402, 16413, 16414, 16416, 16418, 16420],
      dtype='int64', length=4264)

In [48]:
df_new[df_new['PTRACCAT']=='Black']

,PTID,RID,VISCODE,EXAMDATE,AGE,PTGENDER,PTEDUCAT,PTETHCAT,PTRACCAT,PTMARRY,...,RAVLT_immediate_bl,FAQ_bl,Ventricles_bl,Hippocampus_bl,WholeBrain_bl,Entorhinal_bl,Fusiform_bl,MidTemp_bl,ICV_bl,MOCA_bl
34,011_S_0016,16,bl,2005-10-13,65.4,Male,9,Not Hisp/Latino,Black,Married,...,40.0,0.0,17315.0,7309.0,936539.0,3470.0,15931.0,17596.0,1352000.0,NaN
35,011_S_0016,16,m06,2006-04-12,65.4,Male,9,Not Hisp/Latino,Black,Married,...,40.0,0.0,17315.0,7309.0,936539.0,3470.0,15931.0,17596.0,1352000.0,NaN
36,011_S_0016,16,m12,2006-10-11,65.4,Male,9,Not Hisp/Latino,Black,Married,...,40.0,0.0,17315.0,7309.0,936539.0,3470.0,15931.0,17596.0,1352000.0,NaN
37,011_S_0016,16,m24,2007-09-17,65.4,Male,9,Not Hisp/Latino,Black,Married,...,40.0,0.0,17315.0,7309.0,936539.0,3470.0,15931.0,17596.0,1352000.0,NaN
38,011_S_0016,16,m36,2008-10-17,65.4,Male,9,Not Hisp/Latino,Black,Married,...,40.0,0.0,17315.0,7309.0,936539.0,3470.0,15931.0,17596.0,1352000.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16382,114_S_5047,5047,m78,2019-04-25,68.8,Female,16,Not Hisp/Latino,Black,Married,...,29.0,0.0,22131.0,7920.0,1070410.0,4294.0,18238.0,22043.0,1534150.0,24.0
16390,168_S_6908,6908,m30,2023-07-07,89.1,Female,20,Not Hisp/Latino,Black,Divorced,...,31.0,17.0,59975.2,6864.8,939118.0,3454.0,14773.0,17090.0,1374720.0,13.0
16399,035_S_6950,6950,m24,2023-07-18,74.8,Female,18,Not Hisp/Latino,Black,Married,...,36.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.0
16415,041_S_6785,6785,m42,2023-06-15,65.0,Female,18,Not Hisp/Latino,Black,Never married,...,59.0,0.0,20980.7,7282.7,925201.0,3941.0,18164.0,16966.0,1310080.0,30.0


In [58]:
df_new[df_new['AV45'].isna() == False]

,PTID,RID,VISCODE,EXAMDATE,AGE,PTGENDER,PTEDUCAT,PTETHCAT,PTRACCAT,PTMARRY,...,RAVLT_immediate_bl,FAQ_bl,Ventricles_bl,Hippocampus_bl,WholeBrain_bl,Entorhinal_bl,Fusiform_bl,MidTemp_bl,ICV_bl,MOCA_bl
48,082_S_5282,5282,bl,2013-09-09,66.9,Male,17,Not Hisp/Latino,White,Married,...,42.0,0.0,NaN,7851.0,NaN,NaN,NaN,NaN,1498720.0,NaN
49,100_S_5280,5280,m24,2015-09-29,67.5,Male,16,Not Hisp/Latino,Black,Never married,...,42.0,0.0,33185.0,8297.0,1165500.0,4946.0,20147.0,21194.0,1656460.0,28.0
76,100_S_5280,5280,bl,2013-09-17,67.5,Male,16,Not Hisp/Latino,Black,Never married,...,42.0,0.0,33185.0,8297.0,1165500.0,4946.0,20147.0,21194.0,1656460.0,28.0
77,082_S_5279,5279,bl,2013-10-23,68.5,Male,20,Not Hisp/Latino,White,Married,...,61.0,0.0,21327.0,7654.0,1081140.0,4065.0,17964.0,18611.0,1508210.0,27.0
110,067_S_0056,56,m60,2010-12-10,69.6,Female,13,Not Hisp/Latino,Black,Widowed,...,48.0,0.0,13272.0,7606.0,930513.0,2796.0,17161.0,14746.0,1322650.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15818,041_S_5100,5100,m108,2022-02-14,71.2,Male,16,Not Hisp/Latino,White,Married,...,56.0,0.0,53003.0,8542.0,1219990.0,3491.0,19787.0,20413.0,1814640.0,28.0
15822,035_S_6200,6200,m48,2022-03-04,72.3,Male,18,Not Hisp/Latino,White,Married,...,47.0,0.0,46300.1,7301.6,1074240.0,5440.0,19031.0,23556.0,1720000.0,28.0
15830,082_S_6629,6629,m36,2022-01-25,57.0,Female,14,Not Hisp/Latino,Black,Married,...,47.0,0.0,11809.4,7417.2,1016320.0,3091.0,18151.0,22453.0,1322860.0,29.0
15835,041_S_6785,6785,m30,2022-03-01,65.0,Female,18,Not Hisp/Latino,Black,Never married,...,59.0,0.0,20980.7,7282.7,925201.0,3941.0,18164.0,16966.0,1310080.0,30.0
